In [3]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("Vaulty Fraud Detection") \
    .master("local[*]") \
    .getOrCreate()

print("Spark started")

print("Reading csv...")
start_time = time.time()

df = spark.read.csv("work/transactions_20M.csv", header=True, inferSchema=True)

print(f"File read in: {round(time.time() - start_time, 2)} seconds")

df.printSchema()

Spark started
Reading csv...
File read in: 26.55 seconds
root
 |-- transaction_id: string (nullable = true)
 |-- account_from: string (nullable = true)
 |-- account_to: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- is_flagged: boolean (nullable = true)



In [6]:
df.show(5)
start_time = time.time()
total_transactions = df.count()
print(f"Total amount transactions amount: {total_transactions}")
print(f"Time spent: {round(time.time() - start_time, 2)} seconds")

+--------------+------------+------------+-------+-------------------+---------+----------+
|transaction_id|account_from|  account_to| amount|          timestamp|   status|is_flagged|
+--------------+------------+------------+-------+-------------------+---------+----------+
|         TRX-0|SYNTH0043952|SYNTH0046069|2667.21|2019-03-11 03:08:35|  PENDING|     false|
|         TRX-1|SYNTH0000271|SYNTH0040383|4963.62|2020-01-23 03:46:36|  PENDING|     false|
|         TRX-2|SYNTH0026043|SYNTH0044903|4575.61|2021-12-30 15:45:30|   FAILED|     false|
|         TRX-3|SYNTH0018419|SYNTH0046989|1061.15|2022-10-19 00:13:46|   FAILED|     false|
|         TRX-4|SYNTH0041561|SYNTH0045568|3719.55|2020-07-11 00:27:20|COMPLETED|     false|
+--------------+------------+------------+-------+-------------------+---------+----------+
only showing top 5 rows

Total amount transactions amount: 20000000
Time spent: 11.9


In [7]:
import pyspark.sql.functions as F

print("Starting finance audit")
start_time = time.time()

suspicious_df = df.filter((F.col("amount") > 50000) | (F.col("is_flagged") == True))
print(f"Suspicious transactions amount: {suspicious_df.count():,}")

df.groupBy("status").count().show()

df.groupBy("account_from").agg(F.sum("amount").alias("total_sent")).sort(F.desc("total_sent")).show(5)

print(f"Time spent: {round(time.time() - start_time, 2)}")

Suspicious transactions amount: 1,190,961
+---------+--------+
|   status|   count|
+---------+--------+
|   FAILED|  999921|
|COMPLETED|15999794|
|  PENDING| 3000285|
+---------+--------+

+------------+------------------+
|account_from|        total_sent|
+------------+------------------+
|SYNTH0005912|        2533048.12|
|SYNTH0034362|2476753.6900000004|
|SYNTH0040632|        2451497.33|
|SYNTH0012865|2421333.7800000003|
|SYNTH0010638|2396201.7100000004|
+------------+------------------+
only showing top 5 rows

Time spent: 50.59
